# X-DETR — inference + visualization
Load a trained checkpoint and produce the 6-panel gallery for any X-ray image, then
optionally launch the Gradio operator demo.

Run this from the repo root (`jupyter lab` / VS Code) after training with
`python -m engine.train --config configs/xdetr_opixray.yaml`.

In [ ]:
import os

if not os.path.isdir('configs'):
    os.chdir('..')   # notebook was started inside notebooks/

CONFIG = 'configs/xdetr_opixray.yaml'
WEIGHTS = 'runs/opixray_xdetr/final.pth'   # or last.pth / epoch_030.pth

assert os.path.isfile(WEIGHTS), f'no checkpoint at {WEIGHTS} — train first, or point at another'
print('cwd     :', os.getcwd())
print('weights :', WEIGHTS)

In [ ]:
from PIL import Image
from IPython.display import Image as IPyImage

from engine.config import load_config, get_device
from viz.common import load_model
from viz.gallery import make_gallery
from data import build_dataset

cfg = load_config(CONFIG)
device = get_device()
model = load_model(cfg, WEIGHTS, device)
classes = cfg['dataset']['classes']
print(f'loaded on {device}, {len(classes)} classes')

## 1. Six-panel gallery for one test image
input | detections | Eigen-CAM | AIFI encoder saliency | decoder cross-attention | ranked operator map

In [ ]:
os.makedirs('assets', exist_ok=True)
ds = build_dataset(cfg, split='test', train=False)
path = ds.samples[0]['image']          # change the index, or use your own image path
print('image:', path)

make_gallery(model, Image.open(path).convert('RGB'), cfg, device,
             'assets/demo.png', classes, score_thresh=0.3)
IPyImage('assets/demo.png')

## 2. Batch of galleries
The same figure for N test images, written to `assets/galleries/`.

In [ ]:
!python -m scripts.gallery_batch --config {CONFIG} --weights {WEIGHTS} --n 8 --out assets/galleries --score 0.3

import glob
from IPython.display import display
for p in sorted(glob.glob('assets/galleries/*.png'))[:4]:
    display(IPyImage(p))

## 3. Metrics
Per-class AP@0.5, occlusion-stratified mAP (OL1/OL2/OL3), and calibration error (ECE).

In [ ]:
!python -m engine.evaluate --config {CONFIG} --weights {WEIGHTS}

## 4. (Optional) Gradio operator demo
Serves on http://127.0.0.1:7860 — interrupt the cell to stop it.

In [ ]:
!python app/gradio_demo.py --config {CONFIG} --weights {WEIGHTS}